# Interactive Dashboard for Cluster Exploration

# Introduction

Across the last three lessons you built K-Means models, selected features by
variance, scaled them, chose *k* with inertia and silhouette, and visualized the
result with PCA. Every one of those steps required *you* to edit and re-run code.
This lesson removes that requirement. You'll assemble all of it into a live web
application with **Dash**, so anyone — analyst, manager, client — can pick
features, slide the number of clusters, and watch the model and its metrics update
in real time, without touching a line of Python.

🎯 **By the end of this notebook you will be able to:**

-   Build a Dash application layout using HTML and core components.
-   Create callback functions to connect user inputs to dynamic outputs.
-   Integrate K-means clustering and PCA visualization into an interactive
    dashboard.
-   Display model evaluation metrics (inertia and silhouette score) that update in
    real time.

➡️ The lesson splits cleanly in two: first a guided tour of Dash's building blocks
(layout + callbacks) on tiny demos, then a section-by-section build of the real
dashboard around the L3 clustering model.

## Watch first

The video frames the capstone of this project: turning a static analysis into a
tool other people can use. As you watch, notice the shift in audience — every
prior lesson spoke to a data scientist; this one speaks to whoever the data
scientist needs to convince.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1105512918", h="3298dbabb7", width=700, height=450)

# 1. Conceptual Foundation

## Project context: what we're trying to build

The goal is an interactive dashboard that lets users explore consumer-finance
segmentation without writing code. It has three interactive pieces:

1.  A **bar chart** of the five highest-variance features, with a radio button to
    toggle between trimmed and untrimmed variance.
2.  A **slider** to set the number of clusters *k*, with inertia and silhouette
    score that update as you drag it.
3.  A **PCA scatter plot** of the cluster assignments in 2D, reacting to both the
    variance toggle and the cluster count.

🧠 This is the data scientist's communication problem in miniature: the analysis is
done, but its value is zero until a stakeholder can interrogate it themselves. An
interactive tool turns "trust my chart" into "try it yourself."

## Introduction to Dash

Dash is a Python framework for building web apps. Under the hood it stitches
together **Flask** (web server), **React** (frontend), and **Plotly**
(visualizations) — but you write only Python. Every Dash app has two halves:

1.  **Layout** — what the user sees: HTML elements, graphs, sliders, buttons.
2.  **Callbacks** — how the app reacts: Python functions that fire when an input
    changes and update an output.

🧱 Hold onto that layout-vs-callbacks split — it's the organizing idea for the
entire build. Sections 4 builds the layout; Sections 5–7 each add one callback.
Here is the smallest possible Dash app, displaying a single heading:

In [ ]:
from dash import Dash, html

# Create a minimal app (we won't run it yet)
demo_app = Dash(__name__)
demo_app.layout = html.Div([
    html.H1("Hello, Dash!")
])
# To run: demo_app.run(host="0.0.0.0", jupyter_mode="inline")
print("Dash app created successfully. Layout contains:", type(demo_app.layout))

📦 **Key points**

-   [`Dash`](https://dash.plotly.com/minimal-app) — Main Dash application class
-   [`dash.html`](https://dash.plotly.com/dash-html-components) — HTML components
    (Div, H1, H2, H3, etc.)
-   [`dash.dcc`](https://dash.plotly.com/dash-core-components) — Core components
    (Graph, Slider, RadioItems, etc.)

## Building layouts with HTML components

A Dash layout is a *tree* of components. The root is usually an `html.Div`
holding child elements; those children can be more Divs, creating nesting. The
common building blocks:

-   `html.H1`, `html.H2`, `html.H3` — headers of decreasing size
-   `html.Div` — a container for grouping elements
-   `html.P` — paragraph text

🔑 Every component can carry an `id`. That `id` is the wire that connects a
component to a callback later — get it wrong and the callback silently does
nothing. Let's build a small nested layout and inspect its tree:

In [ ]:
from dash import html

# Build a nested layout structure
sample_layout = html.Div([
    html.H1("Main Title"),
    html.H2("Section 1"),
    html.P("This is a paragraph inside section 1."),
    html.Div([
        html.H3("Subsection 1.1"),
        html.P("Nested content goes here.")
    ], id="subsection-container")
])

# Inspect the structure
print("Root element:", type(sample_layout).__name__)
print("Number of children:", len(sample_layout.children))
print("First child type:", type(sample_layout.children[0]).__name__)

📦 **Key points**

-   [`html.Div`](https://dash.plotly.com/dash-html-components/div) — Container for
    grouping elements
-   [`html.H1`](https://dash.plotly.com/dash-html-components/h1) — Level 1 heading

## Interactive components: RadioItems and Slider

Static text isn't enough — the dashboard needs to *capture user input*. Two core
components do that here:

-   `dcc.RadioItems` — a group of radio buttons for picking one option. Its
    `options` is a list of `{"label": ..., "value": ...}` dicts (label is shown,
    value is returned to your callback).
-   `dcc.Slider` — a draggable numeric input defined by `min`, `max`, and `step`.

Each also takes a `value` (the default) and an `id`. Let's create one of each:

In [ ]:
from dash import dcc

# RadioItems example
radio = dcc.RadioItems(
    options=[
        {"label": "Option A", "value": "a"},
        {"label": "Option B", "value": "b"},
        {"label": "Option C", "value": "c"}
    ],
    value="a",  # Default selection
    id="demo-radio"
)
print("RadioItems default value:", radio.value)
print("Number of options:", len(radio.options))

# Slider example
slider = dcc.Slider(
    min=2,
    max=10,
    step=1,
    value=5,  # Default position
    id="demo-slider"
)
print("Slider range:", slider.min, "to", slider.max)
print("Slider default value:", slider.value)

📦 **Key points**

-   [`dcc.RadioItems`](https://dash.plotly.com/dash-core-components/radioitems) —
    Radio button group for single selection
-   [`dcc.Slider`](https://dash.plotly.com/dash-core-components/slider) — Slider
    for numeric input

## Displaying visualizations with dcc.Graph

`dcc.Graph` is the component that renders a Plotly figure inside the page. You give
it an `id`, and a callback later sets its `figure` property to whatever chart you
build. Here's how a Plotly figure slots into a `dcc.Graph`:

In [ ]:
import plotly.express as px
from dash import dcc

# Create a simple Plotly bar chart
sample_data = {"Feature": ["A", "B", "C"], "Value": [10, 25, 15]}
fig = px.bar(x=sample_data["Value"], y=sample_data["Feature"], orientation="h")
fig.update_layout(xaxis_title="Value", yaxis_title="Feature", height=200)

# Wrap it in a dcc.Graph component
graph_component = dcc.Graph(id="sample-chart", figure=fig)
print("Graph component ID:", graph_component.id)
print("Figure has", len(fig.data), "trace(s)")

# Display the figure directly
fig.show()

📦 **Key points**

-   [`dcc.Graph`](https://dash.plotly.com/dash-core-components/graph) — Component
    for displaying Plotly figures
-   [`plotly.express`](https://plotly.com/python/plotly-express/) — High-level
    interface for creating Plotly figures

## Callbacks: connecting inputs to outputs

Layout is the *what*; callbacks are the *how*. A callback is a plain Python
function wearing an `@app.callback` decorator that names its **Inputs** (component
properties that trigger it) and **Outputs** (component properties it updates). When
an input changes, Dash calls your function automatically and pushes the return
value into the output.

🔄 The mental model: **input changes → Dash calls your function → output updates →
page re-renders**. You never call the function yourself; Dash wires it to the UI.
Here's a self-contained example:

In [ ]:
from dash import Dash, Input, Output, dcc, html

# Create a demo app with a slider and text output
callback_demo = Dash(__name__)
callback_demo.layout = html.Div([
    html.H3("Callback Demo"),
    dcc.Slider(min=1, max=10, step=1, value=5, id="input-slider"),
    html.Div(id="output-text")
])

# Define a callback: when slider changes, update the text
@callback_demo.callback(
    Output("output-text", "children"),
    Input("input-slider", "value")
)
def update_text(slider_value):
    return f"The slider value is: {slider_value}"

# Test the callback function directly (without running the server)
result = update_text(7)
print("Callback output for slider_value=7:", result)

A callback can also take *multiple* inputs. The function's parameters line up with
the inputs in the order you list them in the decorator — a detail that matters when
both the radio button and the slider feed the same chart:

In [ ]:
from dash import Input, Output

# Example with multiple inputs (conceptual - not connected to an app)
def demo_multi_input(radio_value, slider_value):
    """Callback with two inputs."""
    return f"Radio: {radio_value}, Slider: {slider_value}"

# Test with sample values
print(demo_multi_input("trimmed", 5))
print(demo_multi_input("not_trimmed", 8))

📦 **Key points**

-   [`dash.Input`](https://dash.plotly.com/basic-callbacks) — Specifies which
    component property triggers the callback
-   [`dash.Output`](https://dash.plotly.com/basic-callbacks) — Specifies which
    component property gets updated

## Recap: K-means clustering and PCA

The dashboard's engine is the exact clustering machinery from L3 — nothing new to
learn, just reassembled inside callbacks. A quick refresher on the scikit-learn
pieces:

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Create toy data for demonstration
toy_data = pd.DataFrame({
    "feature_1": [1, 2, 3, 10, 11, 12, 5, 6],
    "feature_2": [2, 3, 2, 11, 12, 11, 20, 21]
})

# Build a pipeline: scale first, then cluster
pipeline = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=3, random_state=42, n_init=10)
)
pipeline.fit(toy_data)

# Access the fitted KMeans model
kmeans = pipeline.named_steps["kmeans"]
print("Cluster labels:", kmeans.labels_)
print("Inertia:", round(kmeans.inertia_, 2))

# Compute silhouette score
ss = silhouette_score(toy_data, kmeans.labels_)
print("Silhouette score:", round(ss, 3))

And PCA, which collapses the feature space to two components so the clusters can be
plotted:

In [ ]:
from sklearn.decomposition import PCA

# Apply PCA to reduce to 2 dimensions
pca = PCA(n_components=2)
toy_pca = pca.fit_transform(toy_data)

# Create a DataFrame with PCA coordinates and labels
pca_df = pd.DataFrame(toy_pca, columns=["PC1", "PC2"])
pca_df["label"] = kmeans.labels_.astype(str)
print(pca_df)

📦 **Key points**

-   [`sklearn.cluster.KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
    — K-means clustering algorithm
-   [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
    — Principal Component Analysis for dimensionality reduction
-   [`sklearn.pipeline.make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
    — Construct a pipeline from estimators
-   [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
    — Standardize features by removing mean and scaling to unit variance
-   [`sklearn.metrics.silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)
    — Compute the silhouette coefficient for clustering quality

## Computing trimmed variance

The radio button lets users switch between regular and **trimmed** variance for
feature selection. As in L3, trimmed variance drops the extreme tails before
computing, so a few outliers can't hijack the feature ranking:

In [ ]:
import pandas as pd
from scipy.stats.mstats import trimmed_var

# Create toy data with an outlier
toy_series = pd.Series([10, 12, 11, 13, 12, 100])  # 100 is an outlier

# Regular variance (affected by outlier)
regular_var = toy_series.var()
print(f"Regular variance: {regular_var:.2f}")

# Trimmed variance (10% from each tail removed)
trimmed = trimmed_var(toy_series, limits=(0.1, 0.1))
print(f"Trimmed variance (10% limits): {trimmed:.2f}")

# Apply to a DataFrame: compute trimmed variance for each column
toy_df = pd.DataFrame({
    "col_a": [10, 12, 11, 13, 12, 100],
    "col_b": [5, 6, 5, 7, 6, 8]
})
variances = toy_df.apply(trimmed_var, limits=(0.1, 0.1))
print("\nTrimmed variances by column:")
print(variances.sort_values(ascending=False))

📦 **Key points**

-   [`scipy.stats.mstats.trimmed_var`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mstats.trimmed_var.html)
    — Compute variance with trimmed observations

## Common pitfalls and debugging tips

⚠️ Dash apps fail in a handful of predictable ways. When something doesn't work,
check these first:

1.  **Callback not triggering** — the `id` in the callback must match the `id` in
    the layout *exactly*. IDs are case-sensitive.
2.  **Graph not displaying** — your callback must return a valid Plotly figure
    object, not raw data.
3.  **Multiple callbacks updating the same output** — Dash forbids this by
    default; merge the logic into one callback.
4.  **App not launching** — restart the kernel and run all cells from the top.
5.  **Slow updates** — K-Means and PCA are recomputed on every interaction; on
    large data, consider subsampling or caching.

In [ ]:
# Example: verifying callback IDs match layout IDs
layout_ids = {"bar-chart", "trim-button", "k-slider", "metrics", "pca-scatter"}
callback_outputs = {"bar-chart", "metrics", "pca-scatter"}
callback_inputs = {"trim-button", "k-slider"}

# Check all callback references exist in layout
missing = (callback_outputs | callback_inputs) - layout_ids
if missing:
    print(f"Warning: These IDs are missing from layout: {missing}")
else:
    print("All callback IDs are present in layout.")

# Applied Exercises

## 2. Setup

🔧 Gather every import — pandas, Plotly, the Dash pieces, the scikit-learn stack,
and `trimmed_var` — in one cell at the top so all names are in scope before any
component or callback is defined.

**Code 6.4.2.1**:

In [ ]:
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, dcc, html
from scipy.stats.mstats import trimmed_var
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


## 3. Prepare Data

### Problem

Before any dashboard, you need data. The app operates on the same population as
L3: credit-fearful households (`TURNFEAR == 1`) with net worth below \$2 million.
Filtering to this segment keeps the clustering focused and comparable across the
project.

### Approach

Write a `wrangle` function that reads the CSV, applies both filters, and returns
the cleaned DataFrame. It runs once at startup to build the global `df` that every
callback reads from.

📌 As in L2 and L3, `wrangle` is defined in-notebook — P6 keeps a per-lesson copy
rather than importing a shared module.

### Tasks

Define `wrangle` to load the CSV, filter for credit-fearful households with net
worth under \$2 million, and return the cleaned DataFrame.

**Code 6.4.3.1**:

In [ ]:
def wrangle(filepath):
    """Read SCF data file into DataFrame.

    Returns only credit-fearful households whose net worth is less than
    $2 million.

    Parameters
    ----------
    filepath : str
        Location of CSV file.

    Returns
    -------
    pd.DataFrame
    """
    df = pd.read_csv(filepath)
    mask = (df["TURNFEAR"] == 1) & (df["NETWORTH"] < 2e6)
    df = df[mask]
    return df

Load the data with `wrangle` and confirm the shape and a preview of the rows.

**Code 6.4.3.2**:

In [ ]:
df = wrangle("data/SCFP2019.csv.gz")
print(df.shape)
df.head()

### Checkpoint

🧪 These asserts confirm the global `df` the whole dashboard depends on is sound:
non-empty, has the `TURNFEAR` column, every row passes both filters. Since every
callback reads `df`, a failure here would break every chart downstream — so it's
worth getting right before building the UI.

In [ ]:
assert df.shape[0] > 0, "DataFrame is empty after filtering."
assert "TURNFEAR" in df.columns, "Expected column 'TURNFEAR' not found."
assert (df["TURNFEAR"] == 1).all(), (
    "Not all rows have TURNFEAR == 1. "
    f"Found {(df['TURNFEAR'] != 1).sum()} rows with other values."
)
assert (df["NETWORTH"] < 2e6).all(), (
    "Not all rows have NETWORTH < 2,000,000. "
    f"Max NETWORTH found: {df['NETWORTH'].max():,.0f}"
)
print(f"Data loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns.")

## 4. Build Application Layout

### Problem

The first concrete step in any Dash app is the **layout** — the visual skeleton the
user sees. It holds headers, the interactive controls (radio buttons, slider), and
empty placeholders that callbacks will later fill with charts and metrics.

### Approach

Instantiate a `Dash` app and define `app.layout` as an `html.Div` root containing,
in order:

1.  An `H1` header titled "Survey of Consumer Finances".
2.  An `H2` header for "High Variance Features".
3.  A `Graph` for the bar chart (id `"bar-chart"`).
4.  A `RadioItems` to toggle trimmed variance (id `"trim-button"`).
5.  An `H2` header for "K-means Clustering".
6.  An `H3` header for "Number of Clusters (k)".
7.  A `Slider` selecting k from 2 to 12 (id `"k-slider"`).
8.  A `Div` for the metrics (id `"metrics"`).
9.  A `Graph` for the PCA scatter plot (id `"pca-scatter"`).

🔑 Memorize those five ids — `bar-chart`, `trim-button`, `k-slider`, `metrics`,
`pca-scatter`. Every callback in Sections 5–7 references them by name, and a
single typo here is the #1 cause of a silent dashboard.

### Tasks

Create the Dash application instance — the object that will hold the layout and
callbacks.

**Code 6.4.4.1**:

In [ ]:
# Instantiate the Dash application
app = Dash(__name__)

Now define `app.layout` with all nine components: headers, the two graphs, the
radio buttons, the slider, and the metrics div.

**Code Task 6.4.4.2**:

In [ ]:
# Define the application layout
app.layout = html.Div(
    [
        # Headers
        html.H1(...),
        html.H2(...),
        # Bar chart
        dcc.Graph(id=...),
        # Radio buttons for trimmed variance
        dcc.RadioItems(
            options=[
                {"label": "trimmed", "value": True},
                {"label": "not trimmed", "value": False},
            ],
            value=...,
            id=...,
        ),
        # K-means section headers
        html.H2(...),
        html.H3(...),
        # Slider for k
        dcc.Slider(min=..., max=..., step=..., value=..., id=...),
        # Metrics display
        html.Div(id=...),
        # PCA scatter plot
        dcc.Graph(id=...),
    ]
)

📌 Notice the placeholders: `dcc.Graph(id="bar-chart")` and
`dcc.Graph(id="pca-scatter")` carry *no figure yet*, and the metrics `Div` is
empty. The layout only reserves the space — the callbacks you write next are what
actually populate them.

### Checkpoint

🧪 This assert is deliberately loose: it just confirms the layout exists and has at
least nine children. It's a structural smoke test, not a check of every id — those
get exercised when the callbacks run.

In [ ]:
# Verify layout structure
assert app.layout is not None, "app.layout is not defined."
layout_children = app.layout.children
assert len(layout_children) >= 9, (
    f"Expected at least 9 children in layout, got {len(layout_children)}."
)
print("Layout defined successfully with required components.")

## 5. Variance Bar Chart

### Problem

The dashboard's first dynamic piece: a bar chart of the five highest-variance
features, with the radio button letting users switch between trimmed and regular
variance.

### Approach

Two functions, cleanly separated:

1.  `get_high_var_features` — computes variance (trimmed or not) for all columns
    and returns the top five (as a Series, or as names).
2.  `serve_bar_chart` — calls the first to build a horizontal Plotly bar chart.

Then `@app.callback` wires `serve_bar_chart` so the `"trim-button"` input drives
the `"bar-chart"` output.

🧱 This **helper + callback** split repeats in Sections 6 and 7. The helper does the
data/model work and is independently testable; the thin callback just connects it
to the UI. Keep them distinct and the dashboard stays debuggable.

### Tasks

Define `get_high_var_features`: compute variance (trimmed or regular) for all
columns and return the top five.

**Code 6.4.5.1**:

In [ ]:
def get_high_var_features(trimmed=True, return_feat_names=False):
    """Return the five highest-variance features of df.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, calculates trimmed variance, removing bottom and top 10%
        of observations.
    return_feat_names : bool, default=False
        If True, returns feature names as a list. If False, returns a Series
        where index is feature names and values are variances.

    Returns
    -------
    pd.Series or list
    """
    if trimmed:
        top_five = (
            df.apply(trimmed_var, limits=(0.1, 0.1)).sort_values().tail(5)
        )
    else:
        top_five = df.var().sort_values().tail(5)

    if return_feat_names:
        top_five = top_five.index.tolist()

    return top_five

Now write `serve_bar_chart` and decorate it with `@app.callback` to connect the
`"trim-button"` input to the `"bar-chart"` output.

**Code Task 6.4.5.2**:

In [ ]:
@app.callback(Output(..., ...), Input(..., ...))
def serve_bar_chart(trimmed=True):
    """Return a horizontal bar chart of the five highest-variance features.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, calculates trimmed variance.

    Returns
    -------
    plotly.graph_objects.Figure
    """
    top_five = get_high_var_features(...)
    fig = px.bar(x=..., y=..., orientation=...)
    fig.update_layout(xaxis_title=..., yaxis_title=...)
    return fig

### Checkpoint

🧪 Because callbacks are just functions, you can test them directly — no running
server needed. The asserts call `get_high_var_features` and `serve_bar_chart` and
confirm they return a 5-item list and a valid figure. This "call the callback like
a function" trick is the cleanest way to debug Dash logic.

In [ ]:
# Test get_high_var_features
test_features = get_high_var_features(trimmed=True, return_feat_names=True)
assert isinstance(test_features, list), (
    f"Expected list, got {type(test_features).__name__}."
)
assert len(test_features) == 5, (
    f"Expected 5 features, got {len(test_features)}."
)

# Test serve_bar_chart returns a figure
test_fig = serve_bar_chart(trimmed=True)
assert hasattr(test_fig, "data"), "serve_bar_chart did not return a valid figure."
print(f"High variance features (trimmed): {test_features}")
print("Bar chart function working correctly.")

## 6. K-means Metrics

### Problem

When users drag the slider, the dashboard should report fresh evaluation metrics —
inertia and silhouette score — so they can judge cluster quality for each *k* they
try.

### Approach

The same helper + callback pattern:

1.  `get_model_metrics` — builds a `StandardScaler` + `KMeans` pipeline on the top
    five features, fits it, and returns either the model or a metrics dict.
2.  `serve_metrics` — calls it and returns `H3` elements showing the numbers.

`serve_metrics` takes **two** inputs (`"trim-button"` and `"k-slider"`), both
feeding the `"metrics"` output.

### Tasks

Define `get_model_metrics`: build the pipeline, fit it, and return the model or a
dict of metrics.

**Code Task 6.4.6.1**:

In [ ]:
def get_model_metrics(trimmed=True, k=2, return_metrics=False):
    """Build a KMeans model based on the five highest-variance features.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, uses trimmed variance for feature selection.
    k : int, default=2
        Number of clusters.
    return_metrics : bool, default=False
        If False, returns the fitted KMeans pipeline. If True, returns a dict
        with 'inertia' and 'silhouette' keys.

    Returns
    -------
    sklearn.pipeline.Pipeline or dict
    """
    features = get_high_var_features(...)
    X = df[features]
    model = make_pipeline(...)
    model.fit(...)

    if return_metrics:
        inertia = ...
        ss = silhouette_score(...)
        return {"inertia": round(inertia), "silhouette": round(ss, 3)}

    return model

Now write `serve_metrics` with its two inputs, returning `H3` elements that display
inertia and silhouette score.

**Code 6.4.6.2**:

In [ ]:
@app.callback(
    Output("metrics", "children"),
    Input("trim-button", "value"),
    Input("k-slider", "value"),
)
def serve_metrics(trimmed=True, k=2):
    """Return H3 elements displaying inertia and silhouette score.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, uses trimmed variance for feature selection.
    k : int, default=2
        Number of clusters.

    Returns
    -------
    list of html.H3
    """
    metrics = get_model_metrics(trimmed, k, return_metrics=True)
    text = [
        html.H3(f"Inertia: {metrics['inertia']}"),
        html.H3(f"Silhouette Score: {metrics['silhouette']}"),
    ]
    return text

### Checkpoint

🧪 These asserts confirm `get_model_metrics` returns a proper pipeline (with a
`"kmeans"` step) in model mode, and a dict with valid `inertia`/`silhouette` keys
in metrics mode — including that silhouette stays within [−1, 1]. Two return shapes
from one function, both verified.

In [ ]:
# Test get_model_metrics
test_model = get_model_metrics(trimmed=True, k=3, return_metrics=False)
assert hasattr(test_model, "named_steps"), (
    "get_model_metrics should return a Pipeline."
)
assert "kmeans" in test_model.named_steps, (
    "Pipeline should contain a 'kmeans' step."
)

test_metrics = get_model_metrics(trimmed=True, k=3, return_metrics=True)
assert "inertia" in test_metrics, "Metrics dict missing 'inertia' key."
assert "silhouette" in test_metrics, "Metrics dict missing 'silhouette' key."
assert -1 <= test_metrics["silhouette"] <= 1, (
    f"Silhouette score should be in [-1, 1], got {test_metrics['silhouette']}."
)
print(f"Model metrics (k=3): {test_metrics}")
print("Metrics functions working correctly.")

## 7. PCA Scatter Plot

### Problem

The headline visual: a 2D scatter of households colored by cluster. Because the
model uses five features, PCA reduces them to two components so the clusters can be
drawn — and it must re-fit whenever the variance toggle or *k* changes.

### Approach

Helper + callback once more:

1.  `get_pca_labels` — selects the top five features, applies PCA to two
    dimensions, and attaches the cluster labels from a fitted K-Means model.
2.  `serve_scatter_plot` — calls it and builds the Plotly scatter, colored by
    label.

`serve_scatter_plot` takes both inputs and updates the `"pca-scatter"` output.

### Tasks

Define `get_pca_labels`: return a DataFrame of PCA coordinates (`PC1`, `PC2`) plus
cluster labels from the fitted model.

**Code Task 6.4.7.1**:

In [ ]:
def get_pca_labels(trimmed=True, k=2):
    """Return a DataFrame with PCA coordinates and KMeans cluster labels.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, uses trimmed variance for feature selection.
    k : int, default=2
        Number of clusters.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns 'PC1', 'PC2', and 'labels'.
    """
    # Create feature matrix from top 5 features
    features = get_high_var_features(...)
    X = df[features]

    # Apply PCA to reduce to 2 dimensions
    pca = PCA(n_components=...)
    X_pca = pca.fit_transform(...)
    X_pca_df = pd.DataFrame(X_pca, columns=[...])

    # Get cluster labels from fitted model
    model = get_model_metrics(...)
    X_pca_df["labels"] = ...
    X_pca_df.sort_values(..., inplace=True)

    return X_pca_df

Now write `serve_scatter_plot` with two inputs, returning a Plotly scatter colored
by cluster label.

**Code Task 6.4.7.2**:

In [ ]:
@app.callback(
    Output(..., ...),
    Input(..., ...),
    Input(..., ...),
)
def serve_scatter_plot(trimmed=True, k=2):
    """Return a 2D scatter plot of PCA-reduced data with cluster colors.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, uses trimmed variance for feature selection.
    k : int, default=2
        Number of clusters.

    Returns
    -------
    plotly.graph_objects.Figure
    """
    fig = px.scatter(
        data_frame=get_pca_labels(...),
        x=...,
        y=...,
        color=...,
        title=...,
    )
    fig.update_layout(xaxis_title=..., yaxis_title=...)
    return fig

### Checkpoint

🧪 The final component test: `get_pca_labels` returns a DataFrame with `PC1`, `PC2`,
and `labels`, and produces exactly *k* distinct labels (3 here); `serve_scatter_plot`
returns a valid figure. With all three callbacks tested in isolation, you can deploy
with confidence.

In [ ]:
# Test get_pca_labels
test_pca_df = get_pca_labels(trimmed=True, k=3)
assert isinstance(test_pca_df, pd.DataFrame), (
    f"Expected DataFrame, got {type(test_pca_df).__name__}."
)
expected_cols = {"PC1", "PC2", "labels"}
assert expected_cols.issubset(test_pca_df.columns), (
    f"Missing columns. Expected {expected_cols}, got {set(test_pca_df.columns)}."
)
n_unique_labels = test_pca_df["labels"].nunique()
assert n_unique_labels == 3, (
    f"Expected 3 unique labels for k=3, got {n_unique_labels}."
)

# Test serve_scatter_plot
test_scatter = serve_scatter_plot(trimmed=True, k=3)
assert hasattr(test_scatter, "data"), (
    "serve_scatter_plot did not return a valid figure."
)
print(f"PCA DataFrame shape: {test_pca_df.shape}")
print("PCA scatter plot function working correctly.")

## 8. Deploy Application

### Problem

Layout built, all three callbacks defined and tested — the last step is to launch
the server so users can actually interact with the dashboard.

### Approach

Call `app.run()` with a host and port. Inside Jupyter you can pass
`jupyter_mode="external"` to open the app in a new browser tab, or
`jupyter_mode="inline"` to embed it in the notebook.

### Tasks

Run the Dash application. A link will appear to open the dashboard.

**Code 6.4.8.1**:

In [ ]:
app.run(host="0.0.0.0", port=9000, debug=False)

> **Warning:** If you see the error `Address already in use - Port 8050 is in use by another program`, go to **Kernel > Restart Kernel** and then run all cells again from the top.

🚦 More generally, if the app won't launch: (1) restart the kernel and re-run from
the top, (2) make sure no other Dash app is already bound to the same port, and
(3) double-check that every callback id matches a layout id.

### Checkpoint

🧪 There's no programmatic assert for a running server — the real test is
interaction. Toggle the radio button and confirm the bar chart updates; drag the
slider and confirm both the metrics and the scatter plot redraw. If all three
respond, your callbacks are correctly wired.

In [ ]:
# No programmatic checkpoint for app deployment.
# Verify the app is running by interacting with the dashboard:
# 1. Toggle the radio button and confirm the bar chart updates.
# 2. Move the slider and confirm the metrics and scatter plot update.
print("Application deployment complete. Interact with the dashboard to verify.")

# Wrap-up

In this lesson you accomplished the following:

-   Built a complete Dash application layout with headers, graphs, radio buttons,
    and a slider.
-   Created helper functions (`wrangle`, `get_high_var_features`,
    `get_model_metrics`, `get_pca_labels`) to encapsulate data processing and
    modeling logic.
-   Implemented callback functions to connect user inputs to dynamic
    visualizations and metrics.
-   Deployed an interactive dashboard that lets non-technical users explore
    consumer-finance segmentation.

🧠 **The bigger picture — and a caution.** This project closed the loop from raw SCF
records (L1) to a tool a stakeholder can drive themselves (L4). That reach is
powerful, and it carries responsibility. A slider that lets anyone set *k* to any
value will happily produce four "segments" even when the data has no real
structure — the dashboard never refuses. K-Means imposes clusters; it doesn't
discover truths.

⚠️ **Communicate clusters honestly.** When you hand this tool to a non-technical
audience, the silhouette score isn't decoration — it's the guardrail. Pair every
segmentation with its silhouette, flag when separation is weak, and resist
labeling a cluster ("high-risk borrowers") as if it were a verified fact rather
than a statistical grouping. The same interactivity that builds trust can
manufacture false confidence if the metrics are hidden.

➡️ **What's next.** You'll carry these skills — wrangling, modeling, and now
*communicating* results interactively — into new datasets and problems in future
projects, where the audience, not just the algorithm, is part of the design.